In [ ]:
from gensim.models import Word2Vec
import pickle
import numpy as np

# 加载处理后的文本数据
with open("seg_novels.pkl", "rb") as f:
    seg_novels = pickle.load(f)

# 训练Word2Vec模型
model = Word2Vec(sentences=seg_novels, vector_size=100, window=5, min_count=1, sg=1, workers=4, epochs=100)
model.save("word2vec_model.model")

# 加载人物名字列表
with open("person_names.pkl", "rb") as f:
    person_names = pickle.load(f)

# 提取人物向量
person_vectors = []
for name in person_names:
    if name in model.wv.key_to_index:
        person_vectors.append(model.wv[name])
    else:
        print(f"{name} 不在模型词汇表中")

person_vectors = np.array(person_vectors)

# 检查向量维度
print("向量形状：", person_vectors.shape)

# PCA降维到二维和三维
if person_vectors.shape[0] > 1:  # 确保有多个样本
    from sklearn.decomposition import PCA
    pca_2d = PCA(n_components=2)
    pca_3d = PCA(n_components=3)
    person_vectors_2d = pca_2d.fit_transform(person_vectors)
    person_vectors_3d = pca_3d.fit_transform(person_vectors)

    # 保存降维后的向量
    np.save("person_vectors_2d.npy", person_vectors_2d)
    np.save("person_vectors_3d.npy", person_vectors_3d)
else:
    print("样本数量不足，无法进行PCA降维")